# ROI Lens: Advanced Marketing Attribution & Budget Refinement Strategy
**Nexus Consumer Brands Strategy Taskforce**

This notebook contains the complete technical codebase for ROI Lens. The objectives of this platform are:
1. **Bot Traffic Identification & Cleaning**: Audit raw interaction logs to isolate bot activity and calculate wasted budget.
2. **Multi-Touch Probabilistic Attribution**: Implement a Markov Chain model to move away from legacy 'Last-Click' attribution and reveal the true value of each channel.
3. **Financial CPA & ROI Auditing**: Compare legacy Cost Per Acquisition (CPA) with True CPA (excluding bot waste).
4. **Portfolio Reallocation Optimizer**: Calibrate diminishing returns curves and solve budget allocations under Scenario 1 (bots blocked) and Scenario 2 (bots remain).
5. **Brand Persona Analysis**: Integrate Ai Palette trend data and segment insights with buyers to guide campaign targeting.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from scipy.optimize import minimize

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

# Load datasets
print("Loading datasets...")
spend = pd.read_csv("campaign_spend.csv")
profiles = pd.read_csv("user_profiles.csv")
touchpoints = pd.read_csv("touchpoints.csv")

print(f"Campaign Spend: {spend.shape[0]} campaigns loaded.")
print(f"User Profiles: {profiles.shape[0]} users loaded.")
print(f"Touchpoint Logs: {touchpoints.shape[0]} records loaded.")

## Phase 1: Preprocessing & Bot Traffic Detection

We identify bot traffic by checking user-level interaction behavior. Normal users interact occasionally and have low Click-Through Rates (CTR). Bots typically generate very high numbers of events (impressions and clicks) in rapid succession with a perfect 1.0 CTR and 0 conversions.

In [ ]:
# Convert timestamps
touchpoints['Timestamp_dt'] = pd.to_datetime(touchpoints['Timestamp'])
touchpoints['Brand_ID'] = touchpoints['Campaign_ID'].apply(lambda x: x.split('_')[1] if len(x.split('_')) > 1 else 'Unknown')

# Analyze user interaction frequencies
user_counts = touchpoints['User_ID'].value_counts()
print("Top 10 users by event frequency:")
print(user_counts.head(10))

# Identify bots: Users with >= 60 events
bot_users = user_counts[user_counts >= 60].index
touchpoints['Is_Bot'] = touchpoints['User_ID'].isin(bot_users)

print(f"\nDetected {len(bot_users)} bot users (out of {len(user_counts)} total users).")
print(f"Bot touchpoints: {touchpoints['Is_Bot'].sum()} ({touchpoints['Is_Bot'].mean()*100:.2f}% of total)")

# Clean dataset
clean_touchpoints = touchpoints[~touchpoints['Is_Bot']].copy()
clean_touchpoints = clean_touchpoints.sort_values(by=['User_ID', 'Timestamp_dt'])

# Calculate campaign clicks & impressions (total vs clean)
stats = touchpoints.groupby(['Campaign_ID', 'Event_Type']).size().unstack(fill_value=0)
stats_clean = clean_touchpoints.groupby(['Campaign_ID', 'Event_Type']).size().unstack(fill_value=0)

spend_df = spend.copy()
spend_df = spend_df.merge(stats, on='Campaign_ID', how='left').fillna(0)
spend_df = spend_df.rename(columns={'Click': 'Total_Clicks', 'Impression': 'Total_Impressions', 'Purchase': 'Total_Purchases'})
spend_df = spend_df.merge(stats_clean[['Click', 'Impression']], on='Campaign_ID', how='left').fillna(0)
spend_df = spend_df.rename(columns={'Click': 'Clean_Clicks', 'Impression': 'Clean_Impressions'})

# Financial calculations based on Pricing Models
spend_df['Wasted_Fraction'] = np.where(
    spend_df['Pricing_Model'] == 'CPC',
    (spend_df['Total_Clicks'] - spend_df['Clean_Clicks']) / (spend_df['Total_Clicks'] + 1e-5),
    (spend_df['Total_Impressions'] - spend_df['Clean_Impressions']) / (spend_df['Total_Impressions'] + 1e-5)
)

spend_df['Wasted_Budget'] = spend_df['Wasted_Fraction'] * spend_df['Total_Budget_Allocated']
spend_df['Clean_Budget'] = spend_df['Total_Budget_Allocated'] - spend_df['Wasted_Budget']

print(f"\n--- FINANCIAL BOT AUDIT SUMMARY ---")
print(f"Total Budget Allocated: ₹{spend_df['Total_Budget_Allocated'].sum():,.2f}")
print(f"Total Wasted Spend:     ₹{spend_df['Wasted_Budget'].sum():,.2f} ({spend_df['Wasted_Budget'].sum()/spend_df['Total_Budget_Allocated'].sum()*100:.2f}%)")
print(f"Total Clean Spend:      ₹{spend_df['Clean_Budget'].sum():,.2f}")

# Wasted budget by channel
channel_wasted = spend_df.groupby('Channel')[['Total_Budget_Allocated', 'Wasted_Budget']].sum()
channel_wasted['Wasted_Percentage'] = (channel_wasted['Wasted_Budget'] / channel_wasted['Total_Budget_Allocated']) * 100
print("\nWasted Spend by Channel:")
print(channel_wasted)

## Phase 1: Multi-Touch Attribution (MTA) Model

We build a Markov Chain attribution model for each of the 10 brands to sequence user journeys (impressions, clicks, carts) chronologically, ending in either `Purchase` or `Null` (drop-off). Note that the purchase event channel is appended to the journey as the final converting touchpoint. We calculate the Removal Effect for each channel and derive its true attribution weight.

In [ ]:
brands = sorted(clean_touchpoints['Brand_ID'].unique())
if 'Unknown' in brands: brands.remove('Unknown')

def compute_markov_weights(journeys):
    transitions = defaultdict(lambda: defaultdict(int))
    for journey in journeys:
        transitions['Start'][journey[0]] += 1
        for i in range(len(journey) - 1):
            transitions[journey[i]][journey[i+1]] += 1
            
    prob_matrix = defaultdict(dict)
    for state, next_states in transitions.items():
        total_transitions = sum(next_states.values())
        for next_state, count in next_states.items():
            prob_matrix[state][next_state] = count / total_transitions
            
    prob_matrix['Purchase']['Purchase'] = 1.0
    prob_matrix['Null']['Null'] = 1.0
    
    channels = ['Instagram', 'Google Search', 'Influencer Blog', 'YouTube', 'Marketplace']
    all_trans_states = ['Start'] + channels
    state_to_idx = {state: idx for idx, state in enumerate(all_trans_states)}
    num_trans = len(all_trans_states)
    
    Q = np.zeros((num_trans, num_trans))
    R = np.zeros((num_trans, 2))
    
    for state in all_trans_states:
        idx = state_to_idx[state]
        for next_state, prob in prob_matrix[state].items():
            if next_state == 'Purchase':
                R[idx, 0] = prob
            elif next_state == 'Null':
                R[idx, 1] = prob
            elif next_state in state_to_idx:
                Q[idx, state_to_idx[next_state]] = prob
                
    I = np.eye(num_trans)
    try:
        N = np.linalg.inv(I - Q)
        B = N.dot(R)
        base_conversion_prob = B[state_to_idx['Start'], 0]
    except:
        return {c: 0.2 for c in channels}
        
    removal_effects = {}
    for channel in channels:
        Q_mod = Q.copy()
        R_mod = R.copy()
        c_idx = state_to_idx[channel]
        Q_mod[c_idx, :] = 0.0
        R_mod[c_idx, 0] = 0.0
        R_mod[c_idx, 1] = 1.0
        
        try:
            N_mod = np.linalg.inv(I - Q_mod)
            B_mod = N_mod.dot(R_mod)
            mod_conversion_prob = B_mod[state_to_idx['Start'], 0]
            removal_effects[channel] = 1.0 - (mod_conversion_prob / base_conversion_prob)
        except:
            removal_effects[channel] = 0.0
            
    total_re = sum(removal_effects.values())
    if total_re == 0:
        return {c: 0.2 for c in channels}
    return {channel: re / total_re for channel, re in removal_effects.items()}

# Build brand-level journeys and run Markov Chain
brand_markov_weights = {}
brand_purchases_count = {}
brand_last_click_weights = {}

for brand in brands:
    b_tp = clean_touchpoints[clean_touchpoints['Brand_ID'] == brand]
    user_journeys = b_tp.groupby('User_ID')
    
    journeys = []
    last_touch = defaultdict(int)
    total_p = 0
    
    for uid, group in user_journeys:
        events = group.sort_values('Timestamp_dt')
        seq = []
        purchased = False
        for _, row in events.iterrows():
            seq.append(row['Channel'])
            if row['Event_Type'] == 'Purchase':
                purchased = True
                break
        if purchased:
            seq.append('Purchase')
            total_p += 1
            last_touch[seq[-2]] += 1
        else:
            seq.append('Null')
        journeys.append(seq)
        
    weights = compute_markov_weights(journeys)
    brand_markov_weights[brand] = weights
    brand_purchases_count[brand] = total_p
    brand_last_click_weights[brand] = {c: last_touch[c]/total_p for c in ['Instagram', 'Google Search', 'Influencer Blog', 'YouTube', 'Marketplace']} if total_p > 0 else {}

print("Markov Chain Attribution computed successfully for all brands.")

## Phase 1: Legacy CPA vs. True CPA Audit

We audit the cost metrics for each campaign. Legacy CPA divides the entire allocated budget by last-click conversions. True CPA divides the *clean* budget by Markov-attributed conversions, stripping away bot waste and accounting for multi-touch paths.

In [ ]:
def get_m_weight(row):
    return brand_markov_weights[row['Brand_ID']][row['Channel']]

spend_df['Markov_Weight'] = spend_df.apply(get_m_weight, axis=1)

def get_attr_purchases(row):
    brand = row['Brand_ID']
    total_p = brand_purchases_count.get(brand, 0)
    return row['Markov_Weight'] * total_p

spend_df['Attributed_Purchases'] = spend_df.apply(get_attr_purchases, axis=1)
spend_df['Legacy_CPA'] = spend_df['Total_Budget_Allocated'] / (spend_df['Total_Purchases'] + 1e-5)
spend_df['True_CPA'] = spend_df['Clean_Budget'] / (spend_df['Attributed_Purchases'] + 1e-5)

# Show comparison for B01
print("Brand B01 Channel CPA Audit:")
print(spend_df[spend_df['Brand_ID'] == 'B01'][['Channel', 'Total_Budget_Allocated', 'Clean_Budget', 'Wasted_Fraction', 'Markov_Weight', 'Attributed_Purchases', 'Legacy_CPA', 'True_CPA']])

## Phase 2: Budget Reallocation & Diminishing Returns Optimization

We calibrate the diminishing returns curve per channel: $Conversions = k_c \cdot (Spend)^\beta$ with $\beta = 0.75$.
We optimize the ₹10 Crore budget per brand to maximize total conversions under two scenarios:
- **Scenario 1**: Bot traffic blocked (all spend goes to clean traffic).
- **Scenario 2**: Bot traffic remains (wasted CPC/CPM continues).

In [ ]:
BETA = 0.75
BUDGET_LIMIT = 100000000.0 # ₹10 Crore per brand

# Calibrate k_c
spend_df['k_c'] = spend_df['Attributed_Purchases'] / (spend_df['Clean_Budget'] ** BETA + 1e-10)

opt_results = []
for brand in brands:
    brand_df = spend_df[spend_df['Brand_ID'] == brand].copy()
    channels = brand_df['Channel'].tolist()
    k_vals = brand_df['k_c'].tolist()
    wasted_fracs = brand_df['Wasted_Fraction'].tolist()
    prev_budgets = brand_df['Total_Budget_Allocated'].tolist()
    prev_purchases = brand_df['Attributed_Purchases'].tolist()
    
    # SLSQP Solver Functions
    def obj_blocked(x):
        return -sum(k * (val ** BETA) for k, val in zip(k_vals, x))
        
    def obj_remain(x):
        return -sum(k * (((1 - wf) * val) ** BETA) for k, wf, val in zip(k_vals, wasted_fracs, x))
        
    cons = ({'type': 'eq', 'fun': lambda x: sum(x) - BUDGET_LIMIT})
    bounds = [(0, BUDGET_LIMIT) for _ in range(5)]
    x0 = [BUDGET_LIMIT / 5.0] * 5
    
    res_blocked = minimize(obj_blocked, x0, method='SLSQP', bounds=bounds, constraints=cons)
    res_remain = minimize(obj_remain, x0, method='SLSQP', bounds=bounds, constraints=cons)
    
    opt_blocked_spend = res_blocked.x
    opt_remain_spend = res_remain.x
    
    for i, channel in enumerate(channels):
        opt_results.append({
            'Brand_ID': brand,
            'Channel': channel,
            'Prev_Budget': prev_budgets[i],
            'Wasted_Fraction': wasted_fracs[i],
            'Opt_Spend_Bots_Blocked': opt_blocked_spend[i],
            'Opt_Spend_Bots_Remain': opt_remain_spend[i],
            'Expected_Purchases_Bots_Blocked': k_vals[i] * (opt_blocked_spend[i] ** BETA),
            'Expected_Purchases_Bots_Remain': k_vals[i] * (((1 - wasted_fracs[i]) * opt_remain_spend[i]) ** BETA)
        })

opt_df = pd.DataFrame(opt_results)

print(f"Total Historical Conversions: {sum(brand_purchases_count.values())}")
print(f"Scenario 1 (Blocked Bots) Optimized Conversions: {opt_df['Expected_Purchases_Bots_Blocked'].sum():,.2f} (+{ (opt_df['Expected_Purchases_Bots_Blocked'].sum()/sum(brand_purchases_count.values()) - 1)*100:.1f}%)")
print(f"Scenario 2 (Bots Remain) Optimized Conversions: {opt_df['Expected_Purchases_Bots_Remain'].sum():,.2f} (+{ (opt_df['Expected_Purchases_Bots_Remain'].sum()/sum(brand_purchases_count.values()) - 1)*100:.1f}%)")

# Save outputs to CSV
opt_df.to_csv("roi_lens_optimized_allocations.csv", index=False)
print("\nOptimized allocations exported to roi_lens_optimized_allocations.csv")

## Phase 2: Brand Persona & Trend Affinity Integration

We merge buyer segments and trend affinities from `user_profiles.csv` with our purchasing users to identify brand-level target audiences. This allows us to align product positioning and marketing creatives with trend-forecasting results.

In [ ]:
# Merge buyers with profiles
clean_buyers = clean_touchpoints[clean_touchpoints['Event_Type'] == 'Purchase'].copy()
buyers_profile = clean_buyers.merge(profiles, on='User_ID', how='inner')

print(f"Successfully analyzed {buyers_profile.shape[0]} unique buyer profiles.\n")

# Plot overall buyer segments
plt.figure(figsize=(10, 5))
sns.countplot(data=buyers_profile, x='Segment', order=buyers_profile['Segment'].value_counts().index, palette='viridis')
plt.title('Overall Buyer Segment Distribution')
plt.ylabel('Conversion Count')
plt.tight_layout()
plt.show()

# Top personas for B01 and B09
print("Brand B01 Target Segment Profile:")
print(buyers_profile[buyers_profile['Brand_ID'] == 'B01']['Segment'].value_counts(normalize=True).head(3) * 100)

print("\nBrand B09 Target Segment Profile:")
print(buyers_profile[buyers_profile['Brand_ID'] == 'B09']['Segment'].value_counts(normalize=True).head(3) * 100)